### Read file and import libraries

In [54]:
import pandas as pd
import numpy as np

from scipy.optimize import curve_fit, minimize

In [ ]:
file_path = "FINAL_CONTRIBUTIONS.xlsx"

# Read contribution
contribution_df = pd.read_excel(file_path, sheet_name="Contribution")
contribution_df.columns = contribution_df.columns.astype(str).str.strip()
contribution_df = contribution_df.loc[
    :,
    ~contribution_df.columns.str.contains("^Unnamed", case=False, regex=True)
]

contribution_df = contribution_df.rename(columns={
    "base": "base_contribution"
})

contribution_df["Date"] = pd.to_datetime(contribution_df["Date"], errors="coerce")

for col in ["pmax", "meta", "demand_gen", "google_search", "base_contribution"]:
    contribution_df[col] = pd.to_numeric(contribution_df[col], errors="coerce").fillna(0)

contribution_df = contribution_df.dropna(subset=["Date"])
contribution_df = contribution_df.sort_values("Date").reset_index(drop=True)


# Read spends
spend_df = pd.read_excel(file_path, sheet_name="Spends")
spend_df.columns = spend_df.columns.astype(str).str.strip()
spend_df = spend_df.loc[
    :,
    ~spend_df.columns.str.contains("^Unnamed", case=False, regex=True)
]

spend_df["date"] = pd.to_datetime(spend_df["date"], errors="coerce")

for col in [
    "spends_google_demand_gen",
    "spends_google_pmax",
    "spends_google_search",
    "spends_meta"
]:
    spend_df[col] = pd.to_numeric(spend_df[col], errors="coerce").fillna(0)

spend_df = spend_df.dropna(subset=["date"])
spend_df = spend_df.sort_values("date").reset_index(drop=True)


# Final validation dataframe
model_check_df = contribution_df.merge(
    spend_df,
    left_on="Date",
    right_on="date",
    how="inner"
)

model_check_df = model_check_df.drop(columns=["date"])
model_check_df = model_check_df.rename(columns={"Date": "date"})

model_check_df["media_contribution"] = (
    model_check_df["google_search"]
    + model_check_df["pmax"]
    + model_check_df["meta"]
    + model_check_df["demand_gen"]
)

model_check_df["total_predicted_kpi"] = (
    model_check_df["base_contribution"]
    + model_check_df["media_contribution"]
)

model_check_df["total_spend"] = (
    model_check_df["spends_google_search"]
    + model_check_df["spends_google_pmax"]
    + model_check_df["spends_meta"]
    + model_check_df["spends_google_demand_gen"]
)

print("Contribution DF:", contribution_df.shape)
print("Spend DF:", spend_df.shape)
print("Merged model DF:", model_check_df.shape)

display(model_check_df.head())
display(model_check_df.tail())

Contribution DF: (69, 6)
Spend DF: (69, 5)
Merged model DF: (69, 13)


,date,pmax,meta,demand_gen,google_search,base_contribution,spends_google_demand_gen,spends_google_pmax,spends_google_search,spends_meta,media_contribution,total_predicted_kpi,total_spend
0,2024-10-27,0.000000,0.069231,0.0,0.069231,-25.226576,0.0,0.00,1.90,496.35,0.138461,-25.088115,498.25
1,2024-11-03,0.000000,0.000000,0.0,0.000000,-22.621801,0.0,0.00,0.00,2705.43,0.000000,-22.621801,2705.43
2,2024-11-10,1.569650,19.753384,0.0,21.323033,-20.089393,0.0,291.98,500.31,2100.88,42.646066,22.556673,2893.17
3,2024-11-17,9.220067,43.810594,0.0,53.030660,-17.630103,0.0,564.44,1268.01,1010.15,106.061321,88.431218,2842.60
4,2024-11-24,20.794930,22.335750,0.0,43.130681,-15.244659,0.0,577.60,1135.98,843.82,86.261361,71.016702,2557.40


,date,pmax,meta,demand_gen,google_search,base_contribution,spends_google_demand_gen,spends_google_pmax,spends_google_search,spends_meta,media_contribution,total_predicted_kpi,total_spend
64,2026-01-18,36.077826,38.483559,0.0,74.561385,-20.781126,0.0,541.60,1637.78,3992.34,149.122771,128.341645,6171.72
65,2026-01-25,58.247992,33.482588,0.0,91.730581,-23.333584,0.0,1522.72,2365.91,4990.78,183.461162,160.127577,8879.41
66,2026-02-01,66.233647,30.758461,0.0,96.992108,-25.958199,0.0,1801.65,3260.82,5825.42,193.984217,168.026018,10887.89
67,2026-02-08,43.806571,54.293228,0.0,98.099799,-28.654193,0.0,824.45,2745.78,6498.06,196.199597,167.545404,10068.29
68,2026-02-15,3.000860,7.660656,0.0,10.661517,-31.420766,0.0,63.34,344.55,961.14,21.323033,-10.097733,1369.03


### Full updated code with baseline contribution

In [56]:
# ============================================================
# 1. USER CONFIG
# ============================================================

channels = ["google_search", "pmax", "meta", "demand_gen"]

# Contribution columns in your contribution file
contribution_col_map = {
    "google_search": "google_search",
    "pmax": "pmax",
    "meta": "meta",
    "demand_gen": "demand_gen"
}

# Spend columns in your spend file
spend_col_map = {
    "google_search": "spends_google_search",
    "pmax": "spends_google_pmax",
    "meta": "spends_meta",
    "demand_gen": "spends_google_demand_gen"
}

# Baseline / base contribution column
base_contribution_col = "base_contribution"

# Cross-channel synergy / halo pairs
# Keep only synergy pairs with enough historical data
# Demand Gen has very sparse data, so we do not estimate its halo effect yet
synergy_pairs = [
    ("google_search", "pmax"),
    ("google_search", "meta")
]

# Annual planning horizon
n_weeks = 52

# Target uplift scenario
target_uplift = 0.15

In [ ]:
# ============================================================
# BUSINESS GUARDRAILS
# Channel-level min/max budget share
# ============================================================

custom_share_bounds = {
    "google_search": (0, None),  # min 20%, max 45%
    "pmax": (0.10, None),           # min 15%, max 40%
    "meta": (0.20, None),           # min 20%, max 45%
    "demand_gen": (0.00, 0.05)      # max 5% because sparse history
}


cap_60_share_bounds = {
    "google_search": (0.00, 0.60),
    "pmax": (0.10, 0.60),
    "meta": (0.00, 0.60),
    "demand_gen": (0.00, 0.05)
}

cap_70_share_bounds = {
    "google_search": (0.00, 0.70),
    "pmax": (0.10, 0.70),
    "meta": (0.00, 0.70),
    "demand_gen": (0.02, 0.05)
}

In [83]:
def prepare_optimizer_data(contribution_df, spend_df):
    contribution_df = contribution_df.copy()
    spend_df = spend_df.copy()

    contribution_df.columns = contribution_df.columns.str.strip()
    spend_df.columns = spend_df.columns.str.strip()

    # ----------------------------
    # Date handling
    # ----------------------------
    contribution_df["date"] = pd.to_datetime(
        contribution_df["Date"] if "Date" in contribution_df.columns else contribution_df["date"],
        errors="coerce"
    )

    spend_df["date"] = pd.to_datetime(
        spend_df["date"] if "date" in spend_df.columns else spend_df["Date"],
        errors="coerce"
    )

    contribution_df = contribution_df.dropna(subset=["date"])
    spend_df = spend_df.dropna(subset=["date"])

    # ----------------------------
    # Contribution data
    # ----------------------------
    model_df = contribution_df[["date"]].copy()

    for channel in channels:
        contribution_col = contribution_col_map[channel]

        model_df[f"{channel}_contribution"] = pd.to_numeric(
            contribution_df[contribution_col],
            errors="coerce"
        ).fillna(0)

    # ----------------------------
    # Base contribution
    # ----------------------------
    if base_contribution_col in contribution_df.columns:
        model_df["base_contribution"] = pd.to_numeric(
            contribution_df[base_contribution_col],
            errors="coerce"
        ).fillna(0)
    else:
        print(f"Warning: '{base_contribution_col}' not found. Setting base_contribution to 0.")
        model_df["base_contribution"] = 0

    # ----------------------------
    # Spend data
    # ----------------------------
    spend_keep_cols = ["date"] + list(spend_col_map.values())
    spend_df = spend_df[spend_keep_cols].copy()

    for channel in channels:
        spend_col = spend_col_map[channel]

        spend_df[f"{channel}_spend"] = pd.to_numeric(
            spend_df[spend_col],
            errors="coerce"
        ).fillna(0)

    spend_df = spend_df[["date"] + [f"{c}_spend" for c in channels]]

    # ----------------------------
    # Merge contribution + spend
    # ----------------------------
    model_df = model_df.merge(spend_df, on="date", how="inner")

    model_df = model_df.sort_values("date").reset_index(drop=True)

    for channel in channels:
        model_df[f"{channel}_spend"] = model_df[f"{channel}_spend"].fillna(0)
        model_df[f"{channel}_contribution"] = model_df[f"{channel}_contribution"].fillna(0)

    model_df["media_contribution"] = model_df[
        [f"{c}_contribution" for c in channels]
    ].sum(axis=1)

    model_df["total_spend"] = model_df[
        [f"{c}_spend" for c in channels]
    ].sum(axis=1)

    # Final actual KPI from decomposition
    model_df["total_predicted_kpi"] = (
        model_df["base_contribution"] + model_df["media_contribution"]
    )

    return model_df

In [84]:
def response_curve(spend, alpha, beta):
    """
    Diminishing return response curve:
    media contribution = alpha * log(1 + beta * spend)
    """
    return alpha * np.log1p(beta * spend)


def fit_response_curves(model_df):
    response_params = {}

    for channel in channels:
        spend = model_df[f"{channel}_spend"].values.astype(float)
        contribution = model_df[f"{channel}_contribution"].values.astype(float)

        if spend.sum() == 0 or contribution.sum() == 0:
            response_params[channel] = {
                "alpha": 0.0,
                "beta": 0.0,
                "status": "No spend or contribution available"
            }
            continue

        try:
            params, _ = curve_fit(
                response_curve,
                spend,
                contribution,
                bounds=([0, 0], [np.inf, np.inf]),
                maxfev=20000
            )

            response_params[channel] = {
                "alpha": float(params[0]),
                "beta": float(params[1]),
                "status": "Fitted"
            }

        except Exception as e:
            avg_roi = contribution.sum() / spend.sum()

            response_params[channel] = {
                "alpha": float(avg_roi),
                "beta": 1e-6,
                "status": f"Fallback average ROI used: {e}"
            }

    return response_params

In [85]:
def fit_synergy_effects(model_df, response_params):
    """
    We estimate synergy only on media contribution.

    media_contribution =
    individual channel response
    + cross-channel synergy
    """

    base_media_prediction = np.zeros(len(model_df))

    for channel in channels:
        alpha = response_params[channel]["alpha"]
        beta = response_params[channel]["beta"]

        base_media_prediction += response_curve(
            model_df[f"{channel}_spend"].values,
            alpha,
            beta
        )

    media_residual = model_df["media_contribution"].values - base_media_prediction

    synergy_feature_df = pd.DataFrame(index=model_df.index)

    for ch1, ch2 in synergy_pairs:
        feature_name = f"{ch1}__x__{ch2}"

        synergy_feature_df[feature_name] = np.sqrt(
            model_df[f"{ch1}_spend"] * model_df[f"{ch2}_spend"]
        )

    X = synergy_feature_df.values.astype(float)
    y = media_residual.astype(float)

    if X.shape[1] == 0 or np.all(X == 0):
        synergy_params = {
            col: 0.0 for col in synergy_feature_df.columns
        }
        return synergy_params

    # Ridge regularization for stable synergy coefficients
    lambda_reg = 1e-3

    try:
        gamma = np.linalg.inv(
            X.T @ X + lambda_reg * np.eye(X.shape[1])
        ) @ X.T @ y
    except Exception:
        gamma = np.zeros(X.shape[1])

    synergy_params = dict(zip(synergy_feature_df.columns, gamma))

    return synergy_params

In [86]:
def predict_weekly_media_contribution(weekly_spend_vector, response_params, synergy_params):
    spend_dict = dict(zip(channels, weekly_spend_vector))

    media_prediction = 0.0

    # Individual channel response
    for channel in channels:
        alpha = response_params[channel]["alpha"]
        beta = response_params[channel]["beta"]

        media_prediction += response_curve(
            spend_dict[channel],
            alpha,
            beta
        )

    # Cross-channel synergy / halo
    for ch1, ch2 in synergy_pairs:
        key = f"{ch1}__x__{ch2}"

        gamma = synergy_params.get(key, 0.0)

        media_prediction += gamma * np.sqrt(
            spend_dict[ch1] * spend_dict[ch2]
        )

    return float(media_prediction)


def predict_annual_media_contribution(
    annual_spend_vector,
    response_params,
    synergy_params,
    n_weeks=52
):
    weekly_spend_vector = np.array(annual_spend_vector) / n_weeks

    weekly_media_prediction = predict_weekly_media_contribution(
        weekly_spend_vector,
        response_params,
        synergy_params
    )

    return weekly_media_prediction * n_weeks


def predict_annual_total_kpi(
    annual_spend_vector,
    annual_base_contribution,
    response_params,
    synergy_params,
    n_weeks=52
):
    annual_media_contribution = predict_annual_media_contribution(
        annual_spend_vector,
        response_params,
        synergy_params,
        n_weeks=n_weeks
    )

    total_kpi = annual_base_contribution + annual_media_contribution

    return total_kpi

####  Fixed-budget optimizer

The optimizer maximizes only media contribution.

Baseline is added after optimization.

In [87]:
def optimize_fixed_budget(
    total_annual_budget,
    annual_base_contribution,
    response_params,
    synergy_params,
    cap_share=None,
    min_share=0.0,
    custom_share_bounds=None,
    n_weeks=52
):
    """
    Maximize media contribution for a fixed annual budget.
    Then add baseline contribution to report total KPI.

    Priority of constraints:
    1. custom_share_bounds if provided
    2. otherwise generic cap_share / min_share
    """

    n_channels = len(channels)

    bounds = []

    for channel in channels:

        # ----------------------------
        # Use channel-specific bounds
        # ----------------------------
        if custom_share_bounds is not None:
            min_pct, max_pct = custom_share_bounds[channel]

            lower_bound = min_pct * total_annual_budget

            if max_pct is None:
                upper_bound = total_annual_budget
            else:
                upper_bound = max_pct * total_annual_budget

        # ----------------------------
        # Otherwise use generic bounds
        # ----------------------------
        else:
            lower_bound = min_share * total_annual_budget

            if cap_share is not None:
                upper_bound = cap_share * total_annual_budget
            else:
                upper_bound = total_annual_budget

        bounds.append((lower_bound, upper_bound))

    # Initial guess: midpoint of bounds, then scaled to total budget
    x0 = np.array([(low + high) / 2 for low, high in bounds])

    # Scale initial guess to exactly match total budget
    x0 = x0 / x0.sum() * total_annual_budget

    constraints = [
        {
            "type": "eq",
            "fun": lambda x: np.sum(x) - total_annual_budget
        }
    ]

    def objective(x):
        return -predict_annual_media_contribution(
            x,
            response_params,
            synergy_params,
            n_weeks=n_weeks
        )

    result = minimize(
        objective,
        x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={
            "maxiter": 2000,
            "ftol": 1e-9
        }
    )

    optimized_spend = result.x

    optimized_media_contribution = predict_annual_media_contribution(
        optimized_spend,
        response_params,
        synergy_params,
        n_weeks=n_weeks
    )

    optimized_total_kpi = annual_base_contribution + optimized_media_contribution

    output = pd.DataFrame({
        "channel": channels,
        "optimized_annual_spend": optimized_spend
    })

    output["optimized_spend_share"] = (
        output["optimized_annual_spend"] / output["optimized_annual_spend"].sum()
    )

    output["optimized_media_contribution"] = [
        response_curve(
            optimized_spend[i] / n_weeks,
            response_params[channel]["alpha"],
            response_params[channel]["beta"]
        ) * n_weeks
        for i, channel in enumerate(channels)
    ]

    return output, optimized_media_contribution, optimized_total_kpi, result

#### Reverse optimizer

Here also, we optimize only media spend.

But the target can be based on either: media contribution or total KPI (contribution + baseline).

In [88]:
def reverse_optimize_min_spend_for_total_kpi(
    target_annual_total_kpi,
    annual_base_contribution,
    response_params,
    synergy_params,
    max_annual_budget,
    cap_share=None,
    min_share=0.0,
    custom_share_bounds=None,
    n_weeks=52
):
    """
    Reverse optimization.

    Target is total KPI, but only media is optimized.

    Required media contribution =
    target total KPI - baseline contribution
    """

    required_media_contribution = target_annual_total_kpi - annual_base_contribution

    if required_media_contribution <= 0:
        raise ValueError(
            "Target KPI is lower than or equal to baseline contribution. "
            "Media spend is theoretically not required to hit this target."
        )

    bounds = []

    for channel in channels:

        # ----------------------------
        # Use channel-specific bounds
        # For reverse optimization, bounds are based on max_annual_budget
        # ----------------------------
        if custom_share_bounds is not None:
            min_pct, max_pct = custom_share_bounds[channel]

            lower_bound = min_pct * max_annual_budget

            if max_pct is None:
                upper_bound = max_annual_budget
            else:
                upper_bound = max_pct * max_annual_budget

        # ----------------------------
        # Otherwise use generic cap
        # ----------------------------
        else:
            lower_bound = min_share * max_annual_budget

            if cap_share is not None:
                upper_bound = cap_share * max_annual_budget
            else:
                upper_bound = max_annual_budget

        bounds.append((lower_bound, upper_bound))

    # Initial guess: midpoint of bounds
    x0 = np.array([(low + high) / 2 for low, high in bounds])

    def objective(x):
        return np.sum(x)

    constraints = [
        {
            "type": "ineq",
            "fun": lambda x: predict_annual_media_contribution(
                x,
                response_params,
                synergy_params,
                n_weeks=n_weeks
            ) - required_media_contribution
        }
    ]

    result = minimize(
        objective,
        x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={
            "maxiter": 3000,
            "ftol": 1e-9
        }
    )

    optimized_spend = result.x

    optimized_media_contribution = predict_annual_media_contribution(
        optimized_spend,
        response_params,
        synergy_params,
        n_weeks=n_weeks
    )

    optimized_total_kpi = annual_base_contribution + optimized_media_contribution

    output = pd.DataFrame({
        "channel": channels,
        "required_annual_spend": optimized_spend
    })

    output["required_spend_share"] = (
        output["required_annual_spend"] / output["required_annual_spend"].sum()
        if output["required_annual_spend"].sum() > 0
        else 0
    )

    output["required_media_contribution"] = [
        response_curve(
            optimized_spend[i] / n_weeks,
            response_params[channel]["alpha"],
            response_params[channel]["beta"]
        ) * n_weeks
        for i, channel in enumerate(channels)
    ]

    return output, optimized_media_contribution, optimized_total_kpi, result

#### Run all 4 scenarios with baseline

In [89]:
def run_budget_optimizer_scenarios(contribution_df, spend_df):
    model_df = prepare_optimizer_data(contribution_df, spend_df)

    response_params = fit_response_curves(model_df)

    synergy_params = fit_synergy_effects(model_df, response_params)

    # --------------------------------------------------------
    # Annual BAU spend
    # --------------------------------------------------------
    weekly_avg_spend = model_df[[f"{c}_spend" for c in channels]].mean().values
    annual_bau_spend = weekly_avg_spend * n_weeks

    total_annual_bau_budget = annual_bau_spend.sum()

    # --------------------------------------------------------
    # Annual baseline contribution
    # --------------------------------------------------------
    avg_weekly_base_contribution = model_df["base_contribution"].mean()
    annual_base_contribution = avg_weekly_base_contribution * n_weeks


    # --------------------------------------------------------
    # BAU media contribution and total KPI
    # --------------------------------------------------------
    bau_media_contribution = predict_annual_media_contribution(
        annual_bau_spend,
        response_params,
        synergy_params,
        n_weeks=n_weeks
    )

    bau_total_kpi = annual_base_contribution + bau_media_contribution

    # --------------------------------------------------------
    # Scenario 1: BAU 52-week current mix
    # --------------------------------------------------------
    bau_output = pd.DataFrame({
        "channel": channels,
        "bau_annual_spend": annual_bau_spend
    })

    bau_output["bau_spend_share"] = (
        bau_output["bau_annual_spend"] / bau_output["bau_annual_spend"].sum()
    )

    # BAU budget optimized without cap
    (
        bau_optimized_output,
        bau_optimized_media_contribution,
        bau_optimized_total_kpi,
        bau_result
    ) = optimize_fixed_budget(
    total_annual_budget=total_annual_bau_budget,
    annual_base_contribution=annual_base_contribution,
    response_params=response_params,
    synergy_params=synergy_params,
    cap_share=None,
    min_share=0.0,
    custom_share_bounds=custom_share_bounds,
    n_weeks=n_weeks
)

    # --------------------------------------------------------
    # Scenario 2: Existing target
    # Can we spend less to achieve existing BAU total KPI?
    # --------------------------------------------------------
    existing_target_total_kpi = bau_total_kpi

    (
        existing_target_output,
        existing_target_media_contribution,
        existing_target_total_kpi_achieved,
        existing_target_result
    ) = reverse_optimize_min_spend_for_total_kpi(
    target_annual_total_kpi=existing_target_total_kpi,
    annual_base_contribution=annual_base_contribution,
    response_params=response_params,
    synergy_params=synergy_params,
    max_annual_budget=total_annual_bau_budget * 2,
    cap_share=None,
    min_share=0.0,
    custom_share_bounds=custom_share_bounds,
    n_weeks=n_weeks
)

    # --------------------------------------------------------
    # Scenario 3: Total KPI +15%
    # --------------------------------------------------------
    uplift_target_total_kpi = bau_total_kpi * 1.15

    (
        uplift_output,
        uplift_media_contribution,
        uplift_total_kpi_achieved,
        uplift_result
    ) = reverse_optimize_min_spend_for_total_kpi(
        target_annual_total_kpi=uplift_target_total_kpi,
        annual_base_contribution=annual_base_contribution,
        response_params=response_params,
        synergy_params=synergy_params,
        max_annual_budget=total_annual_bau_budget * 3,
        cap_share=None,
        min_share=0.0,
        custom_share_bounds=custom_share_bounds,
        n_weeks=n_weeks
        )

    # --------------------------------------------------------
    # Scenario 4A: Constraint optimize with 60% cap
    # --------------------------------------------------------
    (
        constrained_60_output,
        constrained_60_media_contribution,
        constrained_60_total_kpi,
        constrained_60_result
    ) = optimize_fixed_budget(
    total_annual_budget=total_annual_bau_budget,
    annual_base_contribution=annual_base_contribution,
    response_params=response_params,
    synergy_params=synergy_params,
    custom_share_bounds=cap_60_share_bounds,
    n_weeks=n_weeks
)

    # --------------------------------------------------------
    # Scenario 4B: Constraint optimize with 70% cap
    # --------------------------------------------------------
    (
        constrained_70_output,
        constrained_70_media_contribution,
        constrained_70_total_kpi,
        constrained_70_result
    ) = optimize_fixed_budget(
    total_annual_budget=total_annual_bau_budget,
    annual_base_contribution=annual_base_contribution,
    response_params=response_params,
    synergy_params=synergy_params,
    custom_share_bounds=cap_70_share_bounds,
    n_weeks=n_weeks
    )

    # --------------------------------------------------------
    # Scenario summary
    # --------------------------------------------------------
    summary = pd.DataFrame({
        "scenario": [
            "BAU 52-week current mix",
            "BAU budget optimized without cap",
            "Existing target - minimum spend required",
            "Total KPI +15% - minimum spend required",
            "BAU budget optimized with 60% channel cap",
            "BAU budget optimized with 70% channel cap"
        ],

        "annual_spend": [
            total_annual_bau_budget,
            total_annual_bau_budget,
            existing_target_output["required_annual_spend"].sum(),
            uplift_output["required_annual_spend"].sum(),
            total_annual_bau_budget,
            total_annual_bau_budget
        ],

        "annual_base_contribution": [
            annual_base_contribution,
            annual_base_contribution,
            annual_base_contribution,
            annual_base_contribution,
            annual_base_contribution,
            annual_base_contribution
        ],

        "predicted_media_contribution": [
            bau_media_contribution,
            bau_optimized_media_contribution,
            existing_target_media_contribution,
            uplift_media_contribution,
            constrained_60_media_contribution,
            constrained_70_media_contribution
        ],

        "predicted_total_kpi": [
            bau_total_kpi,
            bau_optimized_total_kpi,
            existing_target_total_kpi_achieved,
            uplift_total_kpi_achieved,
            constrained_60_total_kpi,
            constrained_70_total_kpi
        ]
    })

    summary["spend_vs_bau"] = (
        summary["annual_spend"] - total_annual_bau_budget
    )

    summary["spend_vs_bau_pct"] = (
        summary["spend_vs_bau"] / total_annual_bau_budget
    )

    summary["media_contribution_vs_bau"] = (
        summary["predicted_media_contribution"] - bau_media_contribution
    )

    summary["media_contribution_vs_bau_pct"] = (
        summary["media_contribution_vs_bau"] / bau_media_contribution
    )

    summary["total_kpi_vs_bau"] = (
        summary["predicted_total_kpi"] - bau_total_kpi
    )

    summary["total_kpi_vs_bau_pct"] = (
        summary["total_kpi_vs_bau"] / bau_total_kpi
    )

    # --------------------------------------------------------
    # Parameter outputs
    # --------------------------------------------------------
    response_params_df = (
        pd.DataFrame(response_params)
        .T
        .reset_index()
        .rename(columns={"index": "channel"})
    )

    synergy_params_df = (
        pd.DataFrame.from_dict(
            synergy_params,
            orient="index",
            columns=["synergy_gamma"]
        )
        .reset_index()
        .rename(columns={"index": "synergy_pair"})
    )

    outputs = {
        "model_df": model_df,

        "response_params": response_params_df,
        "synergy_params": synergy_params_df,

        "summary": summary,

        "scenario_1_bau_current_mix": bau_output,
        "scenario_1_bau_optimized_no_cap": bau_optimized_output,
        "scenario_2_existing_target_min_spend": existing_target_output,
        "scenario_3_total_kpi_plus_15_min_spend": uplift_output,
        "scenario_4_constrained_60": constrained_60_output,
        "scenario_4_constrained_70": constrained_70_output,

        "results_status": {
            "bau_optimized_success": bau_result.success,
            "existing_target_success": existing_target_result.success,
            "uplift_target_success": uplift_result.success,
            "constrained_60_success": constrained_60_result.success,
            "constrained_70_success": constrained_70_result.success
        }
    }

    return outputs

In [90]:
outputs = run_budget_optimizer_scenarios(
    contribution_df=contribution_df,
    spend_df=spend_df
)

outputs["summary"]

c:\Users\ShruthiChandrashekar\anaconda3\envs\my_py_3_12\Lib\site-packages\scipy\optimize\_lsq\common.py:115: RuntimeWarning: divide by zero encountered in divide
  phi_prime = -np.sum(suf ** 2 / denom**3) / p_norm
c:\Users\ShruthiChandrashekar\anaconda3\envs\my_py_3_12\Lib\site-packages\scipy\optimize\_slsqp_py.py:437: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)


,scenario,annual_spend,annual_base_contribution,predicted_media_contribution,predicted_total_kpi,spend_vs_bau,spend_vs_bau_pct,media_contribution_vs_bau,media_contribution_vs_bau_pct,total_kpi_vs_bau,total_kpi_vs_bau_pct
0,BAU 52-week current mix,3.611015e+05,511.924858,6931.925440,7443.850299,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
1,BAU budget optimized without cap,3.611015e+05,511.924858,8792.697522,9304.622380,0.000000e+00,0.000000,1860.772081,0.268435,1860.772081,0.249974
2,Existing target - minimum spend required,1.446226e+06,511.924858,6084.580665,6596.505523,1.085125e+06,3.005041,-847.344776,-0.122238,-847.344776,-0.113832
3,Total KPI +15% - minimum spend required,4.678772e+05,511.924858,8048.502985,8560.427844,1.067757e+05,0.295694,1116.577545,0.161078,1116.577545,0.150000
4,BAU budget optimized with 60% channel cap,3.611015e+05,511.924858,8804.714373,9316.639231,0.000000e+00,0.000000,1872.788933,0.270169,1872.788933,0.251589
5,BAU budget optimized with 70% channel cap,3.611015e+05,511.924858,8819.115151,9331.040010,0.000000e+00,0.000000,1887.189711,0.272246,1887.189711,0.253523


In [91]:
outputs["summary"]

,scenario,annual_spend,annual_base_contribution,predicted_media_contribution,predicted_total_kpi,spend_vs_bau,spend_vs_bau_pct,media_contribution_vs_bau,media_contribution_vs_bau_pct,total_kpi_vs_bau,total_kpi_vs_bau_pct
0,BAU 52-week current mix,3.611015e+05,511.924858,6931.925440,7443.850299,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
1,BAU budget optimized without cap,3.611015e+05,511.924858,8792.697522,9304.622380,0.000000e+00,0.000000,1860.772081,0.268435,1860.772081,0.249974
2,Existing target - minimum spend required,1.446226e+06,511.924858,6084.580665,6596.505523,1.085125e+06,3.005041,-847.344776,-0.122238,-847.344776,-0.113832
3,Total KPI +15% - minimum spend required,4.678772e+05,511.924858,8048.502985,8560.427844,1.067757e+05,0.295694,1116.577545,0.161078,1116.577545,0.150000
4,BAU budget optimized with 60% channel cap,3.611015e+05,511.924858,8804.714373,9316.639231,0.000000e+00,0.000000,1872.788933,0.270169,1872.788933,0.251589
5,BAU budget optimized with 70% channel cap,3.611015e+05,511.924858,8819.115151,9331.040010,0.000000e+00,0.000000,1887.189711,0.272246,1887.189711,0.253523


In [92]:
outputs["response_params"]

,channel,alpha,beta,status
0,google_search,127.46127,0.000296,Fitted
1,pmax,15.465505,0.00566,Fitted
2,meta,9.515007,0.005251,Fitted
3,demand_gen,0.066869,6772737431777599666980123540550549408010966944...,Fitted


In [93]:
outputs["synergy_params"]

,synergy_pair,synergy_gamma
0,google_search__x__pmax,-0.024214
1,google_search__x__meta,0.017571


In [94]:
with pd.ExcelWriter("budget_optimizer_outputs_with_baseline.xlsx") as writer:
    outputs["model_df"].to_excel(writer, sheet_name="model_data", index=False)

    outputs["summary"].to_excel(writer, sheet_name="scenario_summary", index=False)

    outputs["response_params"].to_excel(writer, sheet_name="response_curves", index=False)
    outputs["synergy_params"].to_excel(writer, sheet_name="synergy_effects", index=False)

    outputs["scenario_1_bau_current_mix"].to_excel(writer, sheet_name="bau_current_mix", index=False)
    outputs["scenario_1_bau_optimized_no_cap"].to_excel(writer, sheet_name="bau_optimized_no_cap", index=False)
    outputs["scenario_2_existing_target_min_spend"].to_excel(writer, sheet_name="existing_target_min_spend", index=False)
    outputs["scenario_3_total_kpi_plus_15_min_spend"].to_excel(writer, sheet_name="total_kpi_plus_15", index=False)
    outputs["scenario_4_constrained_60"].to_excel(writer, sheet_name="constrained_60_cap", index=False)
    outputs["scenario_4_constrained_70"].to_excel(writer, sheet_name="constrained_70_cap", index=False)